# 00 — Setup
Run this cell at the top of every notebook (Part 0). Mounts Drive, clones/pulls the repo, sets up persistent artifact dirs, installs pinned deps and the CLI RTL toolchain.

In [1]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

REPO = "https://github.com/saiswaroop25-pixel/verilog-slm"  # fill in
if not os.path.exists('/content/verilog-slm'):
    !git clone {REPO} /content/verilog-slm
%cd /content/verilog-slm
!git pull

CKPT = '/content/drive/MyDrive/verilog-slm/checkpoints'
LOGS = '/content/drive/MyDrive/verilog-slm/logs'
os.makedirs(CKPT, exist_ok=True); os.makedirs(LOGS, exist_ok=True)
!ln -sfn {CKPT} artifacts_drive_ckpt
!ln -sfn {LOGS} artifacts_drive_logs

Mounted at /content/drive
Cloning into '/content/verilog-slm'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 58 (delta 0), reused 58 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 69.96 KiB | 1.09 MiB/s, done.
/content/verilog-slm
Already up to date.


In [2]:
# Pinned deps (Part 0 hygiene: Colab silently upgrades packages between sessions)
!pip install -q -r requirements.txt -r requirements-train.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 68.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.4/136.4 kB 10.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 1.26.0 Requires-Python <3.13,>=3.9; 1.26.1 Requires-Python <3.13,>=3.9
ERROR: Could not find a version that satisfies the requirement torch==2.4.1 (from versions: 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0, 2.12.1, 2.13.0, 2.14.0)
ERROR: No matching distribution found for torch==2.4.1


In [ ]:
# requirements-train.txt deliberately doesn't pin torch (Colab ships one
# already matched to the VM's CUDA driver) -- log what's actually here
# instead, per Part 0's "pin every dependency" hygiene.
import torch
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU available:', torch.cuda.is_available())

In [ ]:
import os
# RTL toolchain: iverilog is required (compile+simulate). yosys and verible
# are optional -- the harness soft-gates on them (see docs/industry_standards.md)
# but install them here so the synthesis/lint stages actually run instead of
# being recorded as 'skipped'.
!apt-get -qq update && apt-get -qq install -y iverilog yosys > /dev/null

# verible's release asset filename embeds a version string that changes
# every release (verible-v0.0-NNNN-gHASH-linux-static-x86_64.tar.gz), so a
# fixed "latest/download/<literal-name>" URL goes stale -- resolve the
# actual asset URL via the GitHub API instead. Chained as one shell
# command (not separate `!` lines) so the VERIBLE_URL variable survives
# across the pipe/curl/tar steps -- each `!` line is its own subprocess,
# so a bare shell variable assignment on its own line would silently be
# treated as Python and never reach bash at all.
!VERIBLE_URL=$(curl -s https://api.github.com/repos/chipsalliance/verible/releases/latest \
  | grep -o '"browser_download_url": *"[^"]*linux-static-x86_64.tar.gz"' \
  | head -1 | cut -d'"' -f4) && echo "resolved verible URL: $VERIBLE_URL" && \
  curl -sL "$VERIBLE_URL" -o /tmp/verible.tar.gz && \
  mkdir -p /opt/verible && tar -xzf /tmp/verible.tar.gz -C /opt/verible --strip-components=1

os.environ['PATH'] += ':/opt/verible/bin'
!iverilog -V | head -1
!yosys -V
!verible-verilog-lint --version

In [4]:
# Record the GPU model at the start of every run -- required for the
# per-GPU-hour metric (Part 0 non-negotiable hygiene) to mean anything.
!nvidia-smi -L

/bin/bash: line 1: nvidia-smi: command not found


In [5]:
# Smoke test: the pure-Python side of the pipeline (no GPU/tools needed)
!python -m pytest -q tests/ -x

...................................................                      [100%]
51 passed in 1.35s
